# Agri Vision: EfficientDet-D0, rounds 16–30

Use **4gmail**, select a **T4 GPU**, and **Run all**. This notebook continues only EfficientDet-D0 from the saved round-15 checkpoint through round 30. The round-15 source and its validation-best round-11 checkpoint were verified on 15 October 2026. No new pilot is needed.

Reuse the existing code ZIP and dataset parts. Results go into a separate `_to30_from15` folder. Original training settings are retained: 24 classes, three clients, one local epoch, batch 4, 512-pixel inputs, AdamW and seed 42. Rounds 16–30 retain the existing terminal learning rate of 0.00001; optimizer state resets per client as in the completed run. This is a continuation of the existing schedule.


In [ ]:
from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/Agri Vision")
WORKSPACE = Path("/content/agrivision_30_round")
EXPECTED_DATASET_SHA256 = "7a2920e4a4e0405ee157e9b40ac6306b54d6933b20a306b44a4b7c6871f20868"
EXPECTED_CODE_SHA256 = "a1fe67c876bb3a2e5da123e9f5820652e91189f8fe3b21cead34381c5422f579"
TARGET_ROUNDS = 30
SEED = 42
WORKERS = 2
STAMP = "20261009_224152_449800"
MODELS = ["efficientdet_d0"]
SOURCE_ROOT = DRIVE_ROOT / "comparison_results"
SOURCES = {model: SOURCE_ROOT / f"{model}_non_chili_seed42_{STAMP}_to15" for model in MODELS}
RUNS = {model: SOURCE_ROOT / f"{model}_non_chili_seed42_{STAMP}_to30_from15" for model in MODELS}
EXPECTED_SOURCE_ROUND = 15
EXPECTED_SOURCE_CONFIG_HASH = "5bac396d04beaeab8c865b095b691522fcceba3f4e147d839142a1199be70089"
EXPECTED_SOURCE_CHECKPOINT_SHA256 = "3815ef3cc290b35d529ecd6b2d626a61f1a3ec66ae56e0fd7f2991681c5e391d"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import hashlib, json, shutil, zipfile
def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream,"sha256").hexdigest()
WORKSPACE.mkdir(parents=True, exist_ok=True)
def upload_path(filename):
    path = DRIVE_ROOT / filename
    if path.is_file() or filename != "non_chili_v1_colab_dataset.zip":
        return path
    parts_dir = DRIVE_ROOT / "non_chili_v1_upload_parts"
    manifest_path = parts_dir / "parts.json"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text())
        assert manifest["joined_sha256"] == EXPECTED_DATASET_SHA256, "Wrong dataset parts manifest"
        specs = manifest["parts"]
        assert specs and [item["name"] for item in specs] == [filename + f".part{index:02d}" for index in range(1, len(specs) + 1)], "Missing or incorrectly ordered parts manifest"
        parts = [parts_dir / item["name"] for item in specs]
        assert all(part.is_file() and part.stat().st_size == item["bytes"] for part, item in zip(parts, specs)), "Upload all listed dataset parts with their complete sizes first"
    else:
        parts = [DRIVE_ROOT / (filename + suffix) for suffix in (".part01", ".part02")]
        assert all(part.is_file() for part in parts), "Upload the dataset ZIP or the dataset parts folder first"
    joined = WORKSPACE.parent / filename
    if joined.is_file() and sha(joined) == EXPECTED_DATASET_SHA256:
        return joined
    temporary = joined.with_suffix(joined.suffix + ".partial")
    with temporary.open("wb") as output:
        for part in parts:
            with part.open("rb") as source:
                shutil.copyfileobj(source, output, length=16 * 1024 * 1024)
    assert sha(temporary) == EXPECTED_DATASET_SHA256, "Dataset parts are damaged or do not belong together"
    temporary.replace(joined)
    print(f"Joined {len(parts)} dataset parts on Colab's local disk.")
    return joined
for filename, expected in [("non_chili_v1_colab_dataset.zip", EXPECTED_DATASET_SHA256),
                           ("agrivision_30_round_code.zip", EXPECTED_CODE_SHA256)]:
    path = upload_path(filename)
    assert path.is_file(), f"Upload {filename} first"
    assert sha(path) == expected, f"Wrong or damaged upload: {filename}"
    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            assert (WORKSPACE / name).resolve().is_relative_to(WORKSPACE.resolve()), "Unsafe archive path"
        archive.extractall(WORKSPACE)
print("Verified and extracted both uploads.")


In [ ]:
import os, subprocess, sys
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
# Protect Colab's CUDA-enabled torch/torchvision pair from dependency replacement.
import torch, torchvision
torch_before, vision_before = torch.__version__, torchvision.__version__
assert torch.cuda.is_available(), "Select Runtime -> Change runtime type -> GPU"
gpu = torch.cuda.get_device_name(0)
print("GPU:", gpu, "torch:", torch_before, "torchvision:", vision_before)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-r", "requirements-comparison.txt"], check=True)
# Optional model dependencies, pinned where tested. --no-deps above preserves core CUDA packages.
subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "safetensors", "packaging", "tqdm", "omegaconf==2.4.0",
                "polars>=0.20", "ultralytics-thop>=2.0.18", "matplotlib", "opencv-python", "requests", "scipy", "psutil"], check=True)
assert torch.__version__ == torch_before and torchvision.__version__ == vision_before
for pattern in ["test_comparison.py", "test_30_round_workflow.py"]:
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v"], check=True)


## Verify the completed round-15 source and resume position
Preflight verifies the source weights, configuration, data, original implementation, T4 GPU and recorded package versions. A missing or mismatched round-15 source stops execution before training. After interruption, the new branch resumes from its own latest committed round.


In [ ]:
from scripts.continue_comparison import validate_source
for model in MODELS:
    settings, latest, protocol = validate_source(WORKSPACE, SOURCES[model], torch.device("cuda:0"))
    config = settings["config"]
    assert config["model"] == model and config["seed"] == SEED and config["local_epochs"] == 1
    assert config["rounds"] == EXPECTED_SOURCE_ROUND and latest["round"] == EXPECTED_SOURCE_ROUND, "Expected the completed round-15 source"
    assert settings["config_hash"] == EXPECTED_SOURCE_CONFIG_HASH, "Source configuration differs"
    assert latest["checkpoint_sha256"] == EXPECTED_SOURCE_CHECKPOINT_SHA256, "Wrong round-15 checkpoint"
    assert config["batch"] == 4 and config["optimizer"] == "adamw" and settings["input_size"] == 512
    assert config["lr"] == 0.0001 and config["continuation"]["original_rounds"] == 10
    assert len(protocol["clients"]) == 3 and settings["validation_images"] == 4157
    source_status = json.loads((SOURCES[model] / "status.json").read_text())
    assert source_status["phase"] == "complete" and source_status["completed_rounds"] == 15
    committed = latest["round"]
    if RUNS[model].exists():
        branch = json.loads((RUNS[model] / "latest.json").read_text())
        committed = branch["round"]
        assert EXPECTED_SOURCE_ROUND <= committed <= TARGET_ROUNDS, "Invalid continuation commit"
    if committed == TARGET_ROUNDS:
        print(model, "already completed round 30; no rounds will repeat.")
    else:
        print(model, "resume from committed round:", committed, "next training round:", committed + 1, "target:", TARGET_ROUNDS)
print("Round-15 source and environment preflight checks passed.")


In [ ]:
for model in MODELS:
    print(f"Starting/resuming {model} toward round {TARGET_ROUNDS}", flush=True)
    subprocess.run([sys.executable, "-m", "scripts.continue_comparison", "--root", str(WORKSPACE),
                    "--source", str(SOURCES[model]), "--output", str(RUNS[model]),
                    "--rounds", str(TARGET_ROUNDS), "--workers", str(WORKERS)], check=True)
    print(model, json.loads((RUNS[model] / "latest.json").read_text()))


## Results and automatic resume
The new run is saved under `comparison_results/efficientdet_d0_non_chili_seed42_20261009_224152_449800_to30_from15/`. On its first run, the runner copies metrics for rounds 1–15 and the initial, latest and validation-best checkpoints, then begins round 16. The `_to15` source is retained unchanged.

The runner saves pooled and per-class validation metrics, client logs, and best/latest checkpoints after each round. Check `status.json` for the active phase and `latest.json` for the last fully committed round. The complete history is `results/metrics/global_validation_metrics.csv`; per-class results are in each `round_NNN.json` under `pooled_validation.per_class`. The round-11 best remains selected unless a later round improves pooled mAP50–95. Intermediate new-branch weights are pruned after commits to conserve Drive space; metrics remain.

If Colab stops, reopen this same notebook on 4gmail, select T4 and Run all. Completed rounds are skipped; an unfinished round repeats from the last committed global checkpoint. Training stops after round 30 and starts no other model. Keep both source and continuation folders, code bundle and package versions unchanged. Held-out test evaluation remains separate.
